In [1]:
# ==========================================================
# BASE DE DATOS SUCIA - SIMULACIÓN PRUEBA TÉCNICA FLUID ATTACKS
# Incluye: NULLs, Nulos Implícitos ('N/A', 'UNKNOWN', 'NaN'),
# Duplicados, Inconsistencias de Tipo y Fechas Mal Formateadas.
# ==========================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .appName("FluidAttacks_RealData_Prep") \
    .master("local[*]") \
    .getOrCreate()

# ----------------------------------------------------------
# TABLA 1: proyectos (Con nulos y datos faltantes)
# ----------------------------------------------------------
proyectos_data = [
    (101, "Fintech-API", "Backend", "Alta"),
    (102, "ECommerce-Web", "Frontend", None),        # NULL en criticidad
    (103, "Auth-Service", None, "Crítica"),           # NULL en tipo
    (104, "Data-Pipeline", "Data", "UNKNOWN"),        # Nulo implícito
    (105, "Legacy-Portal", "Web", "Crítica")
]
proyectos_schema = ["id_proyecto", "nombre_proyecto", "tipo", "criticidad_negocio"]

# ----------------------------------------------------------
# TABLA 2: hallazgos (Data Semiautomática de Scanners de Seguridad)
# ----------------------------------------------------------
hallazgos_schema = StructType([
    StructField("id_hallazgo", IntegerType(), True),
    StructField("id_proyecto", IntegerType(), True),
    StructField("vulnerabilidad", StringType(), True),
    StructField("severidad", StringType(), True),
    StructField("cvss_score", DoubleType(), True),    # Nulos explícitos
    StructField("fecha_deteccion", StringType(), True),# Formatos de fecha mixtos
    StructField("fecha_remediacion", StringType(), True),
    StructField("estado", StringType(), True)
])

hallazgos_data = [
    # Normales y resueltos
    (1, 101, "SQL Injection", "CRITICAL", 9.8, "2026-01-10", "2026-01-12", "REMEDIATED"),
    # Sin fecha de remediación ni CVSS (Abierta)
    (2, 101, "XSS Cross-Site Scripting", "HIGH", None, "2026-01-15", None, "OPEN"),
    # Nulo implícito como String ('NULL' o 'N/A')
    (3, 101, "Hardcoded Credentials", "CRITICAL", 8.9, "2026-02-01", "N/A", "REMEDIATED"),
    # Formato de fecha distinto (MM/DD/YYYY)
    (4, 102, "Insecure Dependency", "MEDIUM", 5.3, "01/20/2026", "02/15/2026", "REMEDIATED"),
    # CVSS Nulo
    (5, 102, "XSS Cross-Site Scripting", "MEDIUM", None, "2026-02-10", None, "OPEN"),
    # SLA ultra rápido (0 días)
    (6, 103, "Remote Code Execution", "CRITICAL", 10.0, "2026-01-05", "2026-01-05", "REMEDIATED"),
    # Sin severidad reportada
    (7, 103, "Broken Access Control", None, 8.1, "2026-01-12", None, "IN_PROGRESS"),
    # Duplicado exacto del ID 1 (Filtro requerido)
    (1, 101, "SQL Injection", "CRITICAL", 9.8, "2026-01-10", "2026-01-12", "REMEDIATED"),
    # Duplicado lógico (Diferente ID, mismo evento registrado 2 veces)
    (13, 101, "SQL Injection", "CRITICAL", 9.8, "2026-01-10", "2026-01-12", "REMEDIATED"),
    # Outlier en fecha (Fecha futura o errónea de logs)
    (8, 104, "CSRF", "LOW", 3.2, "2030-01-01", None, "OPEN"),
    # Sin proyecto asignado (Orfano - probar RIGHT/FULL JOINs)
    (9, 999, "Unauthenticated Access", "HIGH", 8.7, "2026-02-01", None, "OPEN")
]

# ----------------------------------------------------------
# TABLA 3: logs_autenticacion (Inyecciones de Brute Force)
# ----------------------------------------------------------
logs_data = [
    (1001, "dev_user1", "192.168.1.10", "SUCCESS", "2026-02-20 08:30:00"),
    (1002, "admin_sec", None, "SUCCESS", "2026-02-20 08:32:00"),             # IP Nula
    (1003, "attacker_x", "45.33.22.11", "FAILED", "2026-02-20 08:35:00"),
    (1004, "attacker_x", "45.33.22.11", "FAILED", "2026-02-20 08:35:10"),
    (1005, "attacker_x", "45.33.22.11", "FAILED", "2026-02-20 08:35:20"),
    (1006, "attacker_x", "45.33.22.11", "SUCCESS", "2026-02-20 08:36:00"),  # Ataque exitoso
    (1007, None, "185.220.101.5", "FAILED", "2026-02-20 09:15:00")            # Usuario Nulo
]
logs_schema = ["id_log", "usuario", "ip_origen", "resultado", "timestamp_acceso"]

# Crear DataFrames
df_proyectos = spark.createDataFrame(proyectos_data, proyectos_schema)
df_hallazgos = spark.createDataFrame(hallazgos_data, hallazgos_schema)
df_logs = spark.createDataFrame(logs_data, logs_schema)

# Crear Vistas SQL
df_proyectos.createOrReplaceTempView("proyectos")
df_hallazgos.createOrReplaceTempView("hallazgos")
df_logs.createOrReplaceTempView("logs_autenticacion")

print("¡Base de Datos REALISTA cargada!")    

¡Base de Datos REALISTA cargada!


---

### Ejercicio 2: Vista proyectos_limpio

Problema: Crea una vista donde los nulos implícitos sean NULL reales, luego reemplázalos por 'Sin clasificar' en tipo y criticidad_negocio. Agrega peso_criticidad numérico (Crítica=4, Alta=3, Media=2, Baja=1, sin clasificar=0).

Pasos/pistas:

Primero normaliza a NULL (CASE/NULLIF) y después aplica COALESCE. Hacerlo en ese orden importa.
Cuidado con la tilde de 'Crítica' al comparar.
Decide qué haces con un peso desconocido y anótalo como comentario.

Por qué: Separar "detectar nulo" de "imputar nulo" evita que un 'UNKNOWN' sobreviva disfrazado de valor válido en un GROUP BY.

In [34]:
query = """
WITH proyectos_limpio AS (
    SELECT
        py.id_proyecto AS id_proyecto,
        TRIM(UPPER(py.nombre_proyecto)) AS nombre_proyecto,
        COALESCE(
            CASE
                WHEN py.tipo IN ('N/A', 'UNKNOWN') THEN NULL
                ELSE py.tipo
            END, 'Sin clasificar') AS tipo,
        COALESCE(
            CASE
                WHEN py.criticidad_negocio IN ('N/A', 'UNKNOWN') THEN NULL
                ELSE py.criticidad_negocio
            END, 'Criticidad negocio') AS criticidad_negocio
    FROM proyectos as py
)

SELECT *
FROM proyectos_limpio
"""
spark.sql(query).show()

+-----------+---------------+--------------+------------------+
|id_proyecto|nombre_proyecto|          tipo|criticidad_negocio|
+-----------+---------------+--------------+------------------+
|        101|    FINTECH-API|       Backend|              Alta|
|        102|  ECOMMERCE-WEB|      Frontend|Criticidad negocio|
|        103|   AUTH-SERVICE|Sin clasificar|           Crítica|
|        104|  DATA-PIPELINE|          Data|Criticidad negocio|
|        105|  LEGACY-PORTAL|           Web|           Crítica|
+-----------+---------------+--------------+------------------+



### Ejercicio 1: Perfil de calidad de hallazgos

Problema: Genera un reporte con una fila por columna de hallazgos que muestre: total de filas, nulos reales, nulos implícitos ('N/A', 'NULL', 'UNKNOWN', 'NaN', '') y el % de completitud.

Pasos/pistas:

Piensa en cómo evitar escribir una query distinta por columna (UNION ALL de un bloque por columna, o stack() para des-pivotar).
Usa COUNT_IF o SUM(CASE WHEN ...) para contar condiciones.
Compara con UPPER(TRIM(col)) para no depender de mayúsculas o espacios.
Recuerda que COUNT(col) ignora NULL y COUNT(*) no.

Por qué: Antes de tocar nada, un analista mide qué tan sucio está el dato. Hacerlo por columna y en porcentaje te da un argumento cuantitativo para justificar cada decisión de limpieza.

In [ ]:
# CReo un CTE para eliminar registros duplicados en la tabla hallazgos, asi garantizo que sus registros no esten duplicados
query = """
-- Elimino registros duplicados en la tabla
WITH c_hallazgos AS (
    SELECT DISTINCT(*)
    FROM hallazgos
),
ha_txt_nrm AS (
    SELECT
        c_ha.id_hallazgo,
        c_ha.id_proyecto,
        COALESCE(TRIM(UPPER(c_ha.vulnerabilidad)), 'UNKNOWN') AS vulnerabilidad,
        COALESCE(TRIM(UPPER(c_ha.severidad)), 'UNKNOWN') AS severidad,
        COALESCE(TRIM(UPPER(c_ha.cvss_score)), 0.0) AS cvss_score,
        COALESCE(
            TRY_TO_DATE(c_ha.fecha_deteccion, 'yyyy-MM-dd'),
            TRY_TO_DATE(c_ha.fecha_deteccion, 'dd/MM/yyyy'),
            TO_DATE ('2000-01-01', 'yyyy-MM-dd')
        ) AS fecha_deteccion,
        COALESCE(
            TRY_TO_DATE(c_ha.fecha_remediacion, 'yyyy-MM-dd'),
            TRY_TO_DATE(c_ha.fecha_remediacion, 'dd/MM/yyyy'),
            TO_DATE('2000-01-01', 'yyyy-MM-dd')
        ) AS fecha_remediacion,
        COALESCE(TRIM(LOWER(c_ha.estado))) AS estado
    FROM c_hallazgos as c_ha
)

SELECT
    
FROM ha_txt_nrm AS ha
"""
spark.sql(query).show()

+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|id_hallazgo|id_proyecto|      vulnerabilidad|severidad|cvss_score|fecha_deteccion|fecha_remediacion|     estado|
+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|          1|        101|       SQL INJECTION| CRITICAL|       9.8|     2026-01-10|       2026-01-12| remediated|
|          3|        101|HARDCODED CREDENT...| CRITICAL|       8.9|     2026-02-01|       2000-01-01| remediated|
|          4|        102| INSECURE DEPENDENCY|   MEDIUM|       5.3|     2000-01-01|       2000-01-01| remediated|
|          6|        103|REMOTE CODE EXECU...| CRITICAL|      10.0|     2026-01-05|       2026-01-05| remediated|
|         13|        101|       SQL INJECTION| CRITICAL|       9.8|     2026-01-10|       2026-01-12| remediated|
|          2|        101|XSS CROSS-SITE SC...|     HIGH|       0.0|     2026-01-15|     